Author: Dr. Pierpaolo Calanna, Phd

## 1. IMPORTS

In [3]:
# imports
import sys
import os
import io
import re
import requests
import json
import datetime
import yaml
import string
import random
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import jinja2 as jn


from pathlib import Path
from base64 import b64encode
from cerberus import Validator
from weasyprint import HTML

# customize matplotlib
matplotlib.rc('font', **{'size' : 8})
matplotlib.use("Agg")

In [4]:
# configuration yaml validator schema
CONF_FILE_SCHEMA = 

# group yaml validator schema
GROUP_FILE_SCHEMA = 

SyntaxError: invalid syntax (4050646271.py, line 2)

## 3. CLASSES

In [5]:
class ABGridValidator():
    def __init__(self):
        # add 
        self.validator = Validator(required_all=True)
        self.conf_file_schema = {
            "titolo": { "type": "string" },
            "numero_gruppi": { "type": "integer", "min": 1, "max": 20 },
            "numero_partecipanti_per_gruppo": { "type": "integer", "min": 3, "max": 12 },
            "consegna": { "type": "string" },
            "domandaA":{ "type": "string" },
            "domandaA_scelte": { "type": "string" },
            "domandaB": { "type": "string" },
            "domandaB_scelte": { "type": "string" },
        }
        self.group_file_schema = {
            "IDGruppo": {
                "type": "integer",
                "min": 1,
                "max": 20
            },
            "scelteA": {
                "type": "list",
                "schema":{
                    "type": "dict",
                    "keysrules": {"type": "string", "regex": "^[A-Z]{1,1}$"},
                    "valuesrules": {"type": "string", "regex": "^([A-Z]{1,1},)*[A-Z]$"}
                }
            },
            "scelteB": {
                "type": "list",
                "schema":{
                    "type": "dict",
                    "keysrules": {"type": "string", "regex": "^[A-Z]{1,1}$"},
                    "valuesrules": {"type": "string", "regex": "^([A-Z]{1,1},)*[A-Z]$"}
                }
            }
        }
        

    def validate(self, type, data):
        schema = self.conf_file_schema if type == "configuration" else self.group_file_schema
        try:
            if self.validator.validate(data, schema):
                # return data and None as errors
                return (True, None)
            # on validation error
            else:
                # return None as data and errors
                return (False, validator.errors)
        except cerberus.DocumentError as e:
            # return None as data and errors
            return (False, "Document was loaded but cannot be evaluated")
        except cerberus.SchemaError as e:
            # return None as data and errors
            return (False, "Invalid yaml validation schema")

### 3.1 NETWORKER CLASS

In [6]:
class Networker():

    def __init__(self, edges):
        # set conversion inch -> cm
        self.cm = 1/2.54

        self.edges = edges
        self.edges_a = self.unpack_edges(edges[0])
        self.nodes_a = set(sum(map(list, edges[0]), []))
        self.edges_b = self.unpack_edges(edges[1])
        self.nodes_b = set(sum(map(list, edges[1]), []))
        
        self.Ga_info = None
        self.Ga_data = None
        self.graphA = None
        self.Gb_info = None
        self.Gb_data = None
        self.graphB = None

    def unpack_edges(self, packed_edges):
        # init edges list
        unpacked_edges = [];
        # loop through rows in data
        for row in packed_edges:
            # loop through nodes and edges
            for node, edges in row.items():
                # split edges
                for edge in edges.split(","):
                    # append edge to edges list
                    unpacked_edges.append((node, edge))
        # return edges list
        return unpacked_edges

    def compute_networks(self):
        # unpack edges
        edges_A, edges_B = self.edges
        # create netowrk A & netowrk B
        Ga, Gb = nx.DiGraph(self.edges_a), nx.DiGraph(self.edges_b)
        # create locations A & locations B
        loca, locb = nx.spring_layout(Ga, k=.5, seed=42), nx.spring_layout(Gb, k=.3, seed=42)
        # set netoworks data
        self.Ga_info, self.Ga_data = self.get_network_stats(Ga)
        self.Gb_info, self.Gb_data = self.get_network_stats(Gb)
        self.graphA = self.get_network_graph(Ga, loca, graphType = "A")
        self.graphB = self.get_network_graph(Gb, locb, graphType = "B")
        
    
    def get_network_graph(self, G, loc, graphType = "A"):
        # define color based of type of network (either A or B)
        color = "#0000FF" if graphType == "A" else "#FF0000"
        # init file buffer
        buffer = io.BytesIO()
        # init plt figure and ax
        fig, ax = plt.subplots(constrained_layout=True, figsize=(9*self.cm,9*self.cm))
        # hide axis
        ax.axis('off')
        #-------------------------------------------------------------------------------------------
        # draw network
        # ------------------------------------------------------------------------------------------
        # draw nodes
        nx.draw_networkx_nodes(G.nodes(), loc, node_color=color, edgecolors=color, ax=ax)
        # store mutual preferences
        mutual_prefs = [ e for e in G.edges() if e[::-1] in G.edges ]
        # store non mutual preferences
        non_mutual_prefs = [ e for e in G.edges if e not in mutual_prefs ]
        # draw mutual preferences
        nx.draw_networkx_edges(G, loc, edgelist=mutual_prefs, edge_color=color, arrowstyle='-', width=3, ax=ax)
        # draw non mutual preferences
        nx.draw_networkx_edges(G, loc, edgelist=non_mutual_prefs, edge_color=color, style="--", arrowstyle='-|>', arrowsize=15, ax=ax)
        # draw labels
        nx.draw_networkx_labels(G, loc, font_color="#FFF", font_weight=True, font_size=14, ax=ax)
        # ------------------------------------------------------------------------------------------
        # save figure to buffer
        fig.savefig(buffer, format="svg", bbox_inches='tight', transparent=True, pad_inches=0.05)
        # close figure
        plt.close(fig)
        # encode buffer to base64
        data = b64encode(buffer.getvalue()).decode()
        # return data as svg uri
        return f"data:image/svg+xml;base64,{data}"
    
    def get_degree_centralization(self, G):
        # to undirected
        Gu = G.to_undirected()
        # determine n
        n = Gu.order()
        # store centrality values
        centrality_values = dict(Gu.degree()).values()
        # determine max degree
        c_max = max(centrality_values)
        # return network centrality
        return sum([ c_max - value for value in centrality_values ]) / ((n-1)*(n-2))
        
    def get_network_stats(self, G):
        # init links dict
        links = dict()
        # init no indegree dict
        no_indegree = dict()
        # loop through nodes
        for node in G.nodes():
            # add x to nodes that do not have indegree, otherwise empty string
            no_indegree[node]="x" if G.in_degree(node) == 0 else ""
            # add joined neighbors
            links[node]=(", ".join(G.neighbors(node)))
        # build stats dataframe
        df = pd.concat([
            pd.Series(links, name="lns"),
            pd.Series(nx.in_degree_centrality(G), name="ic").rank(method="dense", ascending=False),
            pd.Series(nx.pagerank(G, max_iter=1000), name="pr").rank(method="dense", ascending=False),
            pd.Series(nx.betweenness_centrality(G), name="bc").rank(method="dense", ascending=False),
            pd.Series(nx.closeness_centrality(G), name="cc").rank(method="dense", ascending=False),
            pd.Series(
                { n: (len(x)-1)/len(G.nodes()) for n,x in dict(nx.all_pairs_shortest_path_length(G)).items() }, name="or"
            ),
            pd.Series(no_indegree, name="ni")
        ], axis=1)
        # add name to stats dataframe index
        df.index.name = "letter"
        # sort index
        df = df.sort_index()
        # return stats tuple
        return (
            # macro-level stats
            dict(
                nodes = G.number_of_nodes(), 
                edges = G.number_of_edges(),
                degree_centralization = self.get_degree_centralization(G),
                transitivity = nx.transitivity(G),
                reciprocity = nx.reciprocity(G)
            ),
            # micro-level stats
            df
        )

### 3.3 Functions related to DOCUMENTS

In [84]:
class ABGrid():

    def __init__(self, configuration_file, groups_file, templates_path, data_path, sheets_path, reports_path, validator):
        self.configuration_file = Path(configuration_file)
        self.groups_file = [Path(g) for g in groups_file]
        self.prefix = self.configuration_file.stem
        self.templates_path = templates_path
        self.data_path = data_path
        self.sheets_path = sheets_path
        self.reports_path = reports_path
        self.sheet_tpl = "sheet.html"
        self.report_tpl = "report.html"
        self.group_tpl = "group.html"
        self.jinja_env = jn.Environment(loader=jn.FileSystemLoader(self.templates_path))
        self.validator = validator

    def printer_decorator(argument):
        def decorator(function):
            def wrapper(*args, **kwargs):
                print(f"generating {argument}")
                result = function(*args, **kwargs)
                print(f"{argument} generated!")
                return result
            return wrapper
        return decorator
        
    def load_yaml(self, yaml_type, yaml_file):
        # try to load data
        try:
            # open yaml file
            with open(self.data_path / yaml_file, 'r') as file:
                # parse yaml data
                yaml_data = yaml.safe_load(file)
            # if yaml data validates
            validation_status, validation_errors = self.validator.validate(yaml_type, yaml_data)
            if validation_status:
                # return yaml data and None as errors
                return (yaml_data, None)
            # on validation error
            else:
                # return None as data and errors
                return (None, validation_errors)
        # catch exceptions
        except FileNotFoundError:
            # return None as data and errors
            return(None,"Cannot locate files")
        except yaml.YAMLError as e:
            # return None as data and errors
            return (None, "Yaml files could not be parsed")
        
    def get_answersheets_data(self):
        # load configuration data
        yaml_data, validation_errors = self.load_yaml("configuration", self.configuration_file)
        # if configuration data was correctly loaded
        if yaml_data != None:
            # init sheet_data dict
            data = dict()
            # update sheet_data
            data["title"] = yaml_data["titolo"]
            data["groups"] = list(range(1, yaml_data["numero_gruppi"] +1))
            data["likert"] = string.ascii_uppercase[:yaml_data["numero_partecipanti_per_gruppo"]]
            data["explanation"] = yaml_data["consegna"]
            data["ga_question"] = yaml_data["domandaA"]
            data["ga_question_hint"] = yaml_data["domandaA_scelte"]
            data["gb_question"] = yaml_data["domandaB"]
            data["gb_question_hint"] = yaml_data["domandaB_scelte"]
            # return sheet data
            return (data, None)
        # on validation errors
        else:
            # return None and validation errors
            return (None, validation_errors)
        
    def get_report_data(self, group_file):
        # try to load configuration data
        yaml_data, conf_validation_errors = self.load_yaml("configuration", self.configuration_file)
        # if configuration data was correctly loaded
        if yaml_data != None:
            # try to load group data
            group_yaml_data, group_validation_errors = self.load_yaml("group", group_file)
            # if group data was correctly loaded
            if group_yaml_data != None:
                # init networker
                ntw = Networker((group_yaml_data["scelteA"], group_yaml_data["scelteB"]))
                # under this condition
                if len(ntw.nodes_a.symmetric_difference(ntw.nodes_b)) > 0:
                    # return None and errors
                    return (None, "Letters are not correct")
                # compute networks
                ntw.compute_networks()
                # init report data
                report_data = dict()
                # update report data
                report_data["assessment_info"] = yaml_data["titolo"]
                report_data["group_id"] = f'gruppo {group_yaml_data["IDGruppo"]}'
                report_data["ga_question"] = yaml_data["domandaA"]
                report_data["gb_question"] = yaml_data["domandaB"]
                report_data["edges_a"] = ntw.edges_a
                report_data["edges_b"] = ntw.edges_b
                report_data["year"] = datetime.datetime.now(datetime.UTC).year
                # add networks A & B related data to report data
                report_data["ga_info"]  = ntw.Ga_info
                report_data["ga_data"]  = ntw.Ga_data.to_dict("index")
                report_data["ga_graph"] = ntw.graphA
                report_data["gb_info"]  = ntw.Gb_info
                report_data["gb_data"]  = ntw.Gb_data.to_dict("index")
                report_data["gb_graph"] = ntw.graphB
                # return report data
                return (report_data, None)
            # on validation error of group data
            else:
                # return None and validation errors
                return (None, group_validation_errors)
        # on validation error of configuration data
        else:
            # return None and validation errors
            return (None, conf_validation_errors)
    
    def get_data(self, type, *args):
        # retrun sheets data of report data
        return self.get_answersheets_data() if type == "sheets" else self.get_report_data(*args)

    def render_template(self, doc_type, doc_data, doc_template, doc_path, doc_prefix, doc_suffix):
        # try to load sheet template
        try:
            # get doc template
            tpl = self.jinja_env.get_template(doc_template)
            # render doc
            rendered_tpl = tpl.render(doc_data);
            # build file name
            filename_html = re.sub("^_|_$", "", f"{doc_prefix}_{doc_type}_{doc_suffix}.html")
            filename_pdf = re.sub("^_|_$", "", f"{doc_prefix}_{doc_type}_{doc_suffix}.pdf")
            # save doc as pdf
            HTML(string=rendered_tpl).write_pdf(doc_path / filename_pdf)
            # save doc as html
            # with open(path / filename_html, "w") as file: file.write(rendered_tpl)
        # catch exceptions
        except FileNotFoundError:
            return(None, f"Cannot locate {doc_type} template file")

    @printer_decorator("sheets")
    def generate_answer_sheets(self):
        sheets_data, sheets_errors = self.get_data("sheets")
        if sheets_data:
            self.render_template("sheet", sheets_data, self.sheet_tpl, self.sheets_path, self.prefix, '')
        else:
            print(sheets_errors)

    @printer_decorator("groups")
    def generate_group_inputs(self):
        # get doc template
        tpl = self.jinja_env.get_template(self.group_tpl)
        sheets_data, sheets_errors = self.get_data("sheets")
        if (sheets_data != None):
            try:
                # loop thorugh groups
                for g in sheets_data["groups"]:
                    # render current group doc
                    rendered_tpl = re.sub("^\\s*\\n$ ","", tpl.render(sheets_data | { "groupId": g }));
                    # remove blank lines from rendered template
                    rendered_tpl ="\n".join([ line for line in rendered_tpl.split("\n") if len(line)>0 ])
                    # save current group  doc as yaml
                    with open(self.data_path / f"{self.prefix}_gruppo_{g}.yaml", "w") as file:
                        file.write(rendered_tpl)
            # catch exceptions
            except FileNotFoundError:
                print(f"{self.prefix}_gruppo_{g}.yaml file")
        else:
            print(sheets_errors)
    
    @printer_decorator("reports")
    def generate_reports(self):
        # loop through groups
        for group_file in self.groups_file:
            # load report(s) data
            report_data, report_errors = self.get_data("reports", group_file)
            # if report(s) data was correctly loaded
            if (report_data != None):
                # generate report(s)
                self.render_template("report", report_data, self.report_tpl, self.reports_path, "", group_file)
            else:
                # notify user
                print(report_errors)

## 4. SET PROJECT

In [85]:
templates_path = Path("./templates/")
data_path = Path("./data/")
sheets_path = Path("./out/sheets/")
reports_path = Path("./out/reports/")

configuration_file = "2024_sperimentale.yaml"
groups_files = [ "2024_sperimentale_gruppo_1.yaml" ]

abgrid_validator = ABGridValidator()
abgrid = ABGrid(configuration_file, groups_files, templates_path, data_path, sheets_path, reports_path, abgrid_validator)

## 5. GENERATE REPORTS

In [87]:
#abgrid.generate_answer_sheets()
#abgrid.generate_group_inputs()
abgrid.generate_reports()

generating reports
reports generated!
